In [1]:
import torch

print("=" * 50)
print("PYTORCH GPU CHECK")
print("=" * 50)

print("PyTorch version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("GPU unavailable")

print("=" * 50)

PYTORCH GPU CHECK
PyTorch version: 2.11.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
GPU memory: 4.0 GB


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "distilbert"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 96
TRAIN_SAMPLE_SIZE = 10000
VAL_SAMPLE_SIZE = 2000

BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

EPOCHS = 2
LEARNING_RATE = 2e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 50)
print("DISTILBERT TRAINING CONFIGURATION")
print("=" * 50)

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
print("Model:", MODEL_NAME)
print("Max URL length:", MAX_LENGTH)
print("Training samples:", TRAIN_SAMPLE_SIZE)
print("Validation samples:", VAL_SAMPLE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Epochs:", EPOCHS)
print("=" * 50)

DISTILBERT TRAINING CONFIGURATION
Device: cuda
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
Model: distilbert-base-uncased
Max URL length: 96
Training samples: 10000
Validation samples: 2000
Batch size: 2
Gradient accumulation: 8
Epochs: 2


In [3]:
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")

train_sample, _ = train_test_split(
    train_df,
    train_size=TRAIN_SAMPLE_SIZE,
    stratify=train_df["label_binary"],
    random_state=SEED
)

val_sample, _ = train_test_split(
    val_df,
    train_size=VAL_SAMPLE_SIZE,
    stratify=val_df["label_binary"],
    random_state=SEED
)

train_sample = train_sample[["clean_url", "label_binary"]].copy()
val_sample = val_sample[["clean_url", "label_binary"]].copy()

train_sample = train_sample.rename(
    columns={"clean_url": "text", "label_binary": "labels"}
)

val_sample = val_sample.rename(
    columns={"clean_url": "text", "label_binary": "labels"}
)

print("=" * 50)
print("DISTILBERT DATASET")
print("=" * 50)

print("Training URLs:", len(train_sample))
print("Validation URLs:", len(val_sample))

print("\nTraining label distribution:")
print(train_sample["labels"].value_counts().sort_index().to_string())

print("\nValidation label distribution:")
print(val_sample["labels"].value_counts().sort_index().to_string())

print("=" * 50)

DISTILBERT DATASET
Training URLs: 10000
Validation URLs: 2000

Training label distribution:
labels
0    6677
1    3323

Validation label distribution:
labels
0    1335
1     665


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_url = "https://example.com/login/verify-account"

encoded = tokenizer(
    sample_url,
    truncation=True,
    max_length=MAX_LENGTH
)

print("=" * 50)
print("TOKENIZER CHECK")
print("=" * 50)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", len(tokenizer))
print("Sample URL:", sample_url)
print("Token count:", len(encoded["input_ids"]))
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

print("=" * 50)

c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TOKENIZER CHECK
Tokenizer: BertTokenizer
Vocabulary size: 30522
Sample URL: https://example.com/login/verify-account
Token count: 16
Tokens: ['[CLS]', 'https', ':', '/', '/', 'example', '.', 'com', '/', 'log', '##in', '/', 'verify', '-', 'account', '[SEP]']


In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_sample.reset_index(drop=True),
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_sample.reset_index(drop=True),
    preserve_index=False
)

def tokenize_urls(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_urls,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training URLs"
)

val_dataset = val_dataset.map(
    tokenize_urls,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing validation URLs"
)

print("=" * 50)
print("TOKENIZATION COMPLETED")
print("=" * 50)
print("Training URLs:", len(train_dataset))
print("Validation URLs:", len(val_dataset))
print("Dataset columns:", train_dataset.column_names)
print("=" * 50)

Tokenizing validation URLs: 100%|██████████| 2000/2000 [00:00<00:00, 29986.62 examples/s]

TOKENIZATION COMPLETED
Training URLs: 10000
Validation URLs: 2000
Dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Benign", 1: "Malicious"},
    label2id={"Benign": 0, "Malicious": 1}
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("=" * 50)
print("DISTILBERT MODEL LOADED")
print("=" * 50)
print("Model:", MODEL_NAME)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Number of classes:", model.config.num_labels)
print("GPU available:", torch.cuda.is_available())
print("=" * 50)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8331.95it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DISTILBERT MODEL LOADED
Model: distilbert-base-uncased
Total parameters: 66,955,010
Trainable parameters: 66,955,010
Number of classes: 2
GPU available: True


In [7]:
import numpy as np
import inspect

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    probabilities = torch.softmax(
        torch.tensor(logits), dim=-1
    ).numpy()[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probabilities)
    }

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    save_total_limit=1,
    seed=SEED
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics
}

if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("=" * 50)
print("DISTILBERT TRAINER CONFIGURATION")
print("=" * 50)

print("Device:", training_args.device)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Epochs:", EPOCHS)
print("Mixed precision:", training_args.fp16)

print("=" * 50)

DISTILBERT TRAINER CONFIGURATION
Device: cuda:0
Training samples: 10000
Validation samples: 2000
Batch size: 2
Gradient accumulation: 8
Effective batch size: 16
Epochs: 2
Mixed precision: True


In [8]:
import time
import torch

print("=" * 55)
print("STARTING DISTILBERT FINE-TUNING")
print("=" * 55)

print("Device:", training_args.device)
print("Training URLs:", len(train_dataset))
print("Validation URLs:", len(val_dataset))
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

print("\n" + "=" * 55)
print("DISTILBERT TRAINING COMPLETED")
print("=" * 55)

print("Training time (seconds):", round(training_time, 2))
print("Training time (minutes):", round(training_time / 60, 2))
print("Training loss:", round(train_result.training_loss, 4))
print("GPU:", torch.cuda.get_device_name(0))

print("=" * 55)

STARTING DISTILBERT FINE-TUNING
Device: cuda:0
Training URLs: 10000
Validation URLs: 2000
Epochs: 2
Batch size: 2
Effective batch size: 16


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,1.749354,0.144201,0.967000,0.951735,0.948872,0.950301,0.989647
2,1.028078,0.142381,0.972500,0.970679,0.945865,0.958111,0.992096


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.07it/s]



DISTILBERT TRAINING COMPLETED
Training time (seconds): 1074.79
Training time (minutes): 17.91
Training loss: 1.3887
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design


In [9]:
import json
from datetime import datetime

FINAL_MODEL_DIR = MODEL_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

training_summary = {
    "model": MODEL_NAME,
    "training_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "training_time_seconds": round(training_time, 2),
    "training_loss": float(train_result.training_loss),
    "best_validation_f1": float(trainer.state.best_metric),
    "saved_at": datetime.now().isoformat()
}

with open(MODEL_DIR / "training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=4)

print("=" * 55)
print("FINE-TUNED DISTILBERT SAVED")
print("=" * 55)
print("Model location:", FINAL_MODEL_DIR)
print("Training summary:", MODEL_DIR / "training_summary.json")
print("Best validation F1:", round(trainer.state.best_metric, 4))
print("=" * 55)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]

FINE-TUNED DISTILBERT SAVED
Model location: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\final_model
Training summary: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\training_summary.json
Best validation F1: 0.9581


In [10]:
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

test_df = pd.read_csv(DATA_DIR / "test.csv")

test_data = test_df[["clean_url", "label_binary"]].copy()

test_data = test_data.rename(
    columns={
        "clean_url": "text",
        "label_binary": "labels"
    }
)

test_dataset = Dataset.from_pandas(
    test_data.reset_index(drop=True),
    preserve_index=False
)

test_dataset = test_dataset.map(
    tokenize_urls,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing test URLs"
)

print("=" * 55)
print("DISTILBERT TEST EVALUATION")
print("=" * 55)
print("Test samples:", len(test_dataset))
print("Starting test prediction...")

test_start = time.time()

test_output = trainer.predict(test_dataset)

test_time = time.time() - test_start

logits = test_output.predictions
y_true = test_output.label_ids

y_pred = np.argmax(logits, axis=1)

y_prob = torch.softmax(
    torch.tensor(logits), dim=-1
).numpy()[:, 1]

test_metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "inference_time_seconds": round(test_time, 2),
    "test_samples": len(test_dataset),
    "training_samples": len(train_dataset),
    "training_time_seconds": round(training_time, 2)
}

print("\n" + "=" * 55)
print("DISTILBERT FINAL TEST RESULTS")
print("=" * 55)

for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{metric.upper()}: {test_metrics[metric]:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nInference time (seconds):", round(test_time, 2))
print("=" * 55)

Tokenizing test URLs: 100%|██████████| 96167/96167 [00:02<00:00, 36355.75 examples/s]


DISTILBERT TEST EVALUATION
Test samples: 96167
Starting test prediction...



DISTILBERT FINAL TEST RESULTS
ACCURACY: 0.9707
PRECISION: 0.9635
RECALL: 0.9479
F1: 0.9556
ROC_AUC: 0.9925

Confusion Matrix:
[[63063  1148]
 [ 1665 30291]]

Inference time (seconds): 1671.64


In [11]:
RESULTS_DIR = PROJECT_ROOT / "results" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = RESULTS_DIR / "distilbert_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=4)

print("=" * 55)
print("DISTILBERT TEST METRICS SAVED")
print("=" * 55)
print("File:", metrics_path)
print("=" * 55)

DISTILBERT TEST METRICS SAVED
File: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\metrics\distilbert_metrics.json
